<a href="https://colab.research.google.com/github/cidadesdofuturo/Cidades-do-Futuro/blob/main/Graficos_e_indicadores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
#  Gerador de Relatório de Dimensões por Município (FULL HD)
#  Executa no Google Colab
# ============================================================

# ── 1. DEPENDÊNCIAS (Certifique-se de rodar em uma célula antes) ──
!pip install -q playwright openpyxl nest_asyncio
!playwright install chromium
!playwright install-deps chromium

import os, re, html, math, asyncio
import nest_asyncio
import pandas as pd
from google.colab import drive
from IPython.display import display
import ipywidgets as widgets

# Permite rodar o Playwright dentro do loop do Colab
nest_asyncio.apply()

# ── 2. CONFIGURAÇÕES DE CAMINHO ──────────────────────────────
drive.mount('/content/drive', force_remount=True)

BASE_PATH       = "/content/drive/MyDrive/Scripts/Matriz de impacto"
ARQ_INDICADORES = f"{BASE_PATH}/indicadores.xlsx"
OUTPUT_DIR      = f"{BASE_PATH}/Indicadores_graficos"
COL_MUNICIPIO   = "Município"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 3. ESTRUTURA DAS DIMENSÕES (MAPA DE COLUNAS) ──────────────
DIMENSOES = [
    {
        "nome": "Capacidades Institucionais",
        "col_score": 204,
        "topicos": [
            {"nome": "Governança e Planejamento", "col_topico": 211, "indicadores": [("Governança Colaborativa - Responsáveis", 205, 206), ("Incorporação de TICs - Planejamento", 207, 208), ("Planejamento Estratégico para Transformação Digital", 209, 210)]},
            {"nome": "Infraestrutura de TI", "col_topico": 216, "indicadores": [("Governança de TI - Práticas", 212, 213), ("Infraestrutura de Hw e Sw - Armazenamento", 214, 215)]},
            {"nome": "Serviços Públicos Digitais", "col_topico": 223, "indicadores": [("Gestão Integrada de Dados", 217, 218), ("Serviços Públicos On-line", 219, 220), ("Solicitação de Serviços Públicos", 221, 222)]},
            {"nome": "Monitoramento e Transparência", "col_topico": 230, "indicadores": [("Segurança de Políticas Públicas - Monitoramento", 224, 225), ("Percepção dos Serviços Públicos", 226, 227), ("Transparência - Monitoramento", 228, 229)]},
            {"nome": "Dados e Segurança da Informação", "col_topico": 237, "indicadores": [("Transparência - Execução Orçamentária e Financeira", 231, 232), ("Transparência dos Dados - Disponibilização", 233, 234), ("Segurança dos Dados - Práticas", 235, 236)]},
        ],
    },
    {
        "nome": "Econômica",
        "col_score": 30,
        "topicos": [
            {"nome": "Água / Esgoto", "col_topico": 37, "indicadores": [("Índice da população total com atendimento de água", 31, 32), ("Índice da população total com atendimento de esgoto", 33, 34)]},
            {"nome": "Resíduos Sólidos", "col_topico": 42, "indicadores": [("Taxa da população coberta com serviço de coleta de resíduos", 38, 39), ("Coleta seletiva de resíduos no município", 40, 41)]},
            {"nome": "Transporte", "col_topico": 62, "indicadores": [("Serviços regulares de transporte de passageiros", 50, 51), ("Serviços de compartilhamento de viagens", 52, 53)]},
            {"nome": "Vias Públicas", "col_topico": 65, "indicadores": [("Índice de pavimentação das vias públicas", 63, 64)]},
            {"nome": "Conectividade", "col_topico": 80, "indicadores": [("Escala de acesso a banda larga fixa", 66, 67), ("Escala de acesso a banda larga móvel", 68, 69)]},
            {"nome": "Inovação", "col_topico": 89, "indicadores": [("Qualificação profissional e intermediação de mão de obra", 81, 82)]},
            {"nome": "Gestão Urbana", "col_topico": 96, "indicadores": [("Sistema de informação geográfica da prefeitura", 90, 91)]},
            {"nome": "Serviços Online", "col_topico": 99, "indicadores": [("Serviços no website da prefeitura", 97, 98)]},
            {"nome": "Dados Abertos", "col_topico": 102, "indicadores": [("Dados abertos da gestão municipal", 100, 101)]},
        ],
    },
    {
        "nome": "Meio Ambiente",
        "col_score": 174,
        "topicos": [
            {"nome": "Água e Saneamento", "col_topico": 185, "indicadores": [("Índice de volume de esgoto coletado", 175, 176), ("Consumo médio per capita de água", 177, 178), ("Soluções inteligentes para gestão na distribuição e consumo de água", 179, 180), ("Índice de perdas na distribuição de água", 181, 182), ("Índice de volume de esgoto tratado", 183, 184)]},
            {"nome": "Resíduos Sólidos", "col_topico": 190, "indicadores": [("Percentual de material recolhido pela coleta seletiva", 186, 187), ("Soluções inteligentes para otimização da coleta de resíduos", 188, 189)]},
            {"nome": "Áreas Verdes e Meio Ambiente", "col_topico": 193, "indicadores": [("Proteção e gestão do meio ambiente e áreas verdes do município", 191, 192)]},
            {"nome": "Qualidade do Ar e Emissões", "col_topico": 198, "indicadores": [("Soluções em monitoramento de gases de efeito estufa e qualidade do ar", 194, 195), ("Monitoramento da qualidade do ar", 196, 197)]},
            {"nome": "Energia e Iluminação Pública", "col_topico": 203, "indicadores": [("Soluções inteligentes para gestão do consumo de energia elétrica", 199, 200), ("Soluções para telegestão da iluminação pública", 201, 202)]},
        ],
    },
    {
        "nome": "Sociocultural",
        "col_score": 103,
        "topicos": [
            {"nome": "Educação", "col_topico": 122, "indicadores": [("Índice de tecnologia nas escolas", 104, 105), ("Taxa de analfabetismo", 106, 107)]},
            {"nome": "Cultura e Esporte", "col_topico": 131, "indicadores": [("Equipamentos culturais e esportivos", 123, 124), ("Proteção do patrimônio", 125, 126)]},
            {"nome": "Saúde", "col_topico": 146, "indicadores": [("Telemedicina ou telessaúde", 132, 133), ("Leitos hospitalares", 134, 135)]},
            {"nome": "Segurança Pública", "col_topico": 153, "indicadores": [("Monitoramento de segurança", 147, 148), ("Taxa de homicídios", 149, 150)]},
            {"nome": "Defesa Civil", "col_topico": 158, "indicadores": [("Tecnologia para desastres", 154, 155), ("Vulnerabilidade a riscos", 156, 157)]},
            {"nome": "Inclusão Digital", "col_topico": 163, "indicadores": [("Promoção de inclusão digital", 159, 160), ("Capacitação tecnológica", 161, 162)]},
            {"nome": "Inclusão e Equidade", "col_topico": 168, "indicadores": [ ("Inclusão social geral", 166, 167)]},
            {"nome": "Participação Cidadã", "col_topico": 173, "indicadores": [("Participação pública presencial", 169, 170)]},
        ],
    },
]

# ── 4. LÓGICA DE NEGÓCIO E FORMATAÇÃO ───────────────────────

def safe_val(v):
    if v is None or (isinstance(v, float) and math.isnan(v)): return "ND"
    if isinstance(v, float):
        return str(int(v)) if v == int(v) else f"{v:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return str(v)

def nivel_to_int(v):
    try: return int(float(v))
    except: return 1

def nivel_status(n):
    n = nivel_to_int(n)
    if n >= 5: return "status-alto", "ALTO"
    if n >= 3: return "status-medio", "MÉDIO"
    return "status-baixo", "BAIXO"

def top5_altos_baixos(row, topicos):
    indicadores_flat = []
    for t in topicos:
        for nome_ind, col_v, col_n in t["indicadores"]:
            indicadores_flat.append((nome_ind, nivel_to_int(row.iloc[col_n])))
    fortes = [n for n, _ in sorted(indicadores_flat, key=lambda x: -x[1])[:5]]
    atencao = [n for n, _ in sorted(indicadores_flat, key=lambda x: x[1])[:5]]
    return fortes, atencao

# ── 5. TEMPLATES HTML/CSS ───────────────────────────────────

CSS = """
* { box-sizing: border-box; -webkit-print-color-adjust: exact; }
body { font-family: 'Open Sans', sans-serif; background: #ECECEC; margin: 0; padding: 0; color: #222; }
.page { width: 960px; margin: 20px auto; background: #FFF; padding: 40px; border-radius: 4px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); }
.title { font-family: 'Montserrat', sans-serif; font-size: 28px; font-weight: 800; margin-bottom: 5px; color: #111; }
.subtitle { font-size: 14px; color: #666; margin-bottom: 25px; border-left: 4px solid #F5C200; padding-left: 10px; }
.section-title { font-family: 'Montserrat', sans-serif; font-size: 12px; font-weight: 800; text-transform: uppercase; letter-spacing: 0.1em; margin: 25px 0 15px; padding-bottom: 5px; border-bottom: 2px solid #F5C200; display: inline-block; }
.bar-row { display: flex; align-items: center; gap: 12px; margin-bottom: 12px; }
.bar-name { width: 250px; font-weight: 600; font-size: 12px; }
.bar-track { flex: 1; height: 12px; background: #EEE; border-radius: 10px; overflow: hidden; }
.bar-fill { height: 100%; background: #F5C200; border-radius: 10px; }
.bar-score { width: 30px; text-align: right; font-weight: 800; font-size: 14px; }
.two-col { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-top: 25px; }
.box { padding: 15px; border-radius: 8px; border: 1px solid #ddd; }
.pos { background: #F2F9F6; border-color: #B0DBC2; }
.neg { background: #FFF8F8; border-color: #F5B8BB; }
.box h3 { margin: 0 0 10px; font-family: 'Montserrat', sans-serif; font-size: 13px; text-transform: uppercase; color: #333; }
.box ul { margin: 0; padding-left: 0; list-style: none; }
.box li { margin-bottom: 6px; font-size: 12px; position: relative; padding-left: 18px; line-height: 1.4; }
.pos li::before { content: '✓'; position: absolute; left: 0; color: #0F6E56; font-weight: bold; }
.neg li::before { content: '!'; position: absolute; left: 0; color: #A32D2D; font-weight: bold; }
table { width: 100%; border-collapse: collapse; margin-top: 10px; font-size: 12px; }
th, td { padding: 12px 10px; border-bottom: 1px solid #EEE; text-align: left; }
th { background: #111; color: #F5C200; font-family: 'Montserrat', sans-serif; text-transform: uppercase; font-size: 10px; }
.status-badge { padding: 4px 10px; border-radius: 20px; font-size: 9px; font-weight: 800; text-transform: uppercase; }
.status-alto { background: #DCEEE7; color: #0F6E56; }
.status-medio { background: #F7E8C9; color: #9A6400; }
.status-baixo { background: #F6DCDD; color: #A32D2D; }
.overview-page { width: 960px; margin: 20px auto; background: #FFF; padding: 40px; border-radius: 4px; box-shadow: 0 4px 10px rgba(0,0,0,0.1); }
.dim-card { display: flex; align-items: center; gap: 20px; padding: 20px; border-radius: 10px; border: 1px solid #ddd; margin-bottom: 16px; background: #FAFAFA; }
.dim-icon { width: 60px; height: 60px; border-radius: 50%; background: #F5C200; display: flex; align-items: center; justify-content: center; font-family: 'Montserrat', sans-serif; font-size: 22px; font-weight: 800; color: #111; flex-shrink: 0; }
.dim-info { flex: 1; }
.dim-name { font-family: 'Montserrat', sans-serif; font-size: 15px; font-weight: 800; color: #111; margin-bottom: 6px; }
.dim-bar-track { width: 100%; height: 14px; background: #EEE; border-radius: 10px; overflow: hidden; }
.dim-bar-fill { height: 100%; border-radius: 10px; }
.dim-score { font-family: 'Montserrat', sans-serif; font-size: 32px; font-weight: 800; color: #111; min-width: 60px; text-align: right; }
.dim-score span { font-size: 14px; color: #888; font-weight: 400; }
.media-box { margin-top: 30px; padding: 24px 30px; border-radius: 12px; background: #111; display: flex; align-items: center; justify-content: space-between; }
.media-label { font-family: 'Montserrat', sans-serif; font-size: 14px; font-weight: 700; text-transform: uppercase; letter-spacing: 0.1em; color: #F5C200; }
.media-valor { font-family: 'Montserrat', sans-serif; font-size: 48px; font-weight: 800; color: #FFF; }
.media-valor span { font-size: 18px; color: #AAA; font-weight: 400; }
.media-barra-track { flex: 1; margin: 0 30px; height: 16px; background: #333; border-radius: 10px; overflow: hidden; }
.media-barra-fill { height: 100%; background: #F5C200; border-radius: 10px; }
"""

def gerar_html(municipio, estado, row):
    pages = []
    for dim in DIMENSOES:
        score = nivel_to_int(row.iloc[dim["col_score"]])
        fortes, atencao = top5_altos_baixos(row, dim["topicos"])

        # Página 1: Resumo
        barras = "".join([f'<div class="bar-row"><span class="bar-name">{html.escape(t["nome"])}</span><div class="bar-track"><div class="bar-fill" style="width:{nivel_to_int(row.iloc[t["col_topico"]])/7*100}%"></div></div><span class="bar-score">{nivel_to_int(row.iloc[t["col_topico"]])}</span></div>' for t in dim["topicos"]])
        p1 = f"""<div class="page"><div class="title">{html.escape(dim['nome'])}</div><div class="subtitle">{html.escape(municipio)} - {html.escape(estado)} | Pontuação: {score}/7</div><div class="section-title">Visão por Tópico</div>{barras}<div class="two-col"><div class="box pos"><h3>Pontos Fortes</h3><ul>{''.join([f'<li>{html.escape(x)}</li>' for x in fortes])}</ul></div><div class="box neg"><h3>Pontos de Atenção</h3><ul>{''.join([f'<li>{html.escape(x)}</li>' for x in atencao])}</ul></div></div></div>"""

        # Página 2: Detalhes
        linhas = ""
        for t in dim["topicos"]:
            for i, (n_ind, _c_v, c_n) in enumerate(t["indicadores"]):
                nv = nivel_to_int(row.iloc[c_n])
                cls, lbl = nivel_status(nv)
                td_topic = f'<td rowspan="{len(t["indicadores"])}" style="font-weight:700; background:#F9F9F9; width:180px;">{html.escape(t["nome"])}</td>' if i == 0 else ""
                linhas += f'<tr>{td_topic}<td>{html.escape(n_ind)}</td><td style="font-weight:800;">{nv}</td><td><span class="status-badge {cls}">{lbl}</span></td></tr>'
        p2 = f"""<div class="page"><div class="title">{html.escape(dim['nome'])}</div><div class="section-title">Indicadores Detalhados</div><table><thead><tr><th>Tópico</th><th>Indicador</th><th>Nível</th><th>Status</th></tr></thead><tbody>{linhas}</tbody></table></div>"""

        pages.extend([p1, p2])

    return f"<!DOCTYPE html><html><head><meta charset='utf-8'/><link href='https://fonts.googleapis.com/css2?family=Montserrat:wght@700;800&family=Open+Sans:wght@400;600&display=swap' rel='stylesheet'/><style>{CSS}</style></head><body>{''.join(pages)}</body></html>"


DIM_COLORS = {
    "Capacidades Institucionais": "#4A90D9",
    "Econômica": "#27AE60",
    "Meio Ambiente": "#16A085",
    "Sociocultural": "#8E44AD"
}

def gerar_html_overview(municipio, estado, row):
    """Gera uma página HTML com o resumo geral das 4 dimensões e a média."""
    cards = ""
    scores = []
    for dim in DIMENSOES:
        score = nivel_to_int(row.iloc[dim["col_score"]])
        scores.append(score)
        cor = DIM_COLORS.get(dim["nome"], "#F5C200")
        pct = score / 7 * 100
        cls, lbl = nivel_status(score)
        cards += f'''
        <div class="dim-card">
          <div class="dim-icon" style="background:{cor}; color:#FFF;">{score}</div>
          <div class="dim-info">
            <div class="dim-name">{html.escape(dim["nome"])}</div>
            <div class="dim-bar-track">
              <div class="dim-bar-fill" style="width:{pct:.1f}%; background:{cor};"></div>
            </div>
          </div>
          <div class="dim-score">{score}<span>/7</span></div>
          <span class="status-badge {cls}">{lbl}</span>
        </div>'''

    media = sum(scores) / len(scores)
    media_pct = media / 7 * 100
    cls_media, lbl_media = nivel_status(round(media))

    overview_html = f'''
    <div class="overview-page">
      <div class="title">Resumo Geral — {html.escape(municipio)}</div>
      <div class="subtitle">{html.escape(estado)} | Avaliação por Dimensão (escala 1–7)</div>
      <div class="section-title">Notas por Dimensão</div>
      {cards}
      <div class="media-box">
        <div>
          <div class="media-label">Média Geral das 4 Dimensões</div>
          <div style="margin-top:8px;"><span class="status-badge {cls_media}" style="font-size:12px;">{lbl_media}</span></div>
        </div>
        <div class="media-barra-track">
          <div class="media-barra-fill" style="width:{media_pct:.1f}%;"></div>
        </div>
        <div class="media-valor">{media:.2f}<span>/7</span></div>
      </div>
    </div>'''
    return f"<!DOCTYPE html><html><head><meta charset='utf-8'/><link href='https://fonts.googleapis.com/css2?family=Montserrat:wght@700;800&family=Open+Sans:wght@400;600&display=swap' rel='stylesheet'/><style>{CSS}</style></head><body>{overview_html}</body></html>"


# ── 6. CAPTURA DE IMAGENS (PLAYWRIGHT) ──────────────────────

async def capturar_png_async(html_path, municipio, dim_nomes):
    from playwright.async_api import async_playwright
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        # device_scale_factor=2 garante Alta Resolução (DPI alto)
        context = await browser.new_context(viewport={"width": 1100, "height": 1400}, device_scale_factor=2)
        page = await context.new_page()
        await page.goto(f"file://{html_path}", wait_until="networkidle")

        elementos = await page.query_selector_all(".page")
        mun_slug = re.sub(r"[^a-zA-Z0-9]", "_", municipio)

        for i, el in enumerate(elementos):
            dim_idx = i // 2
            tipo = "01_Resumo" if i % 2 == 0 else "02_Tabela"
            dim_slug = re.sub(r"[^a-zA-Z0-9]", "_", dim_nomes[dim_idx])

            nome_arq = f"{mun_slug}_{dim_slug}_{tipo}.png"
            await el.screenshot(path=os.path.join(OUTPUT_DIR, nome_arq), scale="device")
            print(f"  ✓ {nome_arq} gerado.")

        await browser.close()

async def capturar_png_overview_async(html_path, municipio):
    from playwright.async_api import async_playwright
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        context = await browser.new_context(viewport={"width": 1100, "height": 900}, device_scale_factor=2)
        page = await context.new_page()
        await page.goto(f"file://{html_path}", wait_until="networkidle")

        el = await page.query_selector(".overview-page")
        mun_slug = re.sub(r"[^a-zA-Z0-9]", "_", municipio)
        nome_arq = f"{mun_slug}_00_Resumo_Geral.png"
        await el.screenshot(path=os.path.join(OUTPUT_DIR, nome_arq), scale="device")
        print(f"  ✓ {nome_arq} gerado.")
        await browser.close()

# ── 7. EXECUÇÃO PRINCIPAL ───────────────────────────────────

print("Carregando base de dados...")
df = pd.read_excel(ARQ_INDICADORES, sheet_name="Indicadores")
municipios = sorted(df[COL_MUNICIPIO].dropna().unique().tolist())

def gerar_relatorio(b):
    with output_area:
        output_area.clear_output()
        mun_nome = dropdown.value
        mask = df[COL_MUNICIPIO].str.strip() == mun_nome
        if not mask.any(): return print("Erro: Município não encontrado.")

        row = df[mask].iloc[0]
        estado = row.get("Estado", "BR")
        html_str = gerar_html(mun_nome, estado, row)

        tmp_path = os.path.abspath("temp.html")
        with open(tmp_path, "w", encoding="utf-8") as f: f.write(html_str)

        print(f"🏙️ Processando {mun_nome}...")
        asyncio.run(capturar_png_async(tmp_path, mun_nome, [d["nome"] for d in DIMENSOES]))

        # Gerar imagem de resumo geral (notas das 4 dimensões + média)
        html_overview = gerar_html_overview(mun_nome, estado, row)
        tmp_overview = os.path.abspath("temp_overview.html")
        with open(tmp_overview, "w", encoding="utf-8") as f: f.write(html_overview)
        asyncio.run(capturar_png_overview_async(tmp_overview, mun_nome))

        print(f"✨ Pronto! Arquivos em: {OUTPUT_DIR}")

# ── 8. INTERFACE DO COLAB ───────────────────────────────────

dropdown = widgets.Combobox(options=municipios, placeholder="Escolha um município", description="📍", layout={'width': '400px'})
btn = widgets.Button(description="Gerar Gráficos PNG", button_style="success", icon="image")
output_area = widgets.Output()

btn.on_click(gerar_relatorio)
display(widgets.VBox([widgets.HTML("<h2>Gerador de Gráficos de Indicadores</h2>"), dropdown, btn, output_area]))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 13.7 MB/s eta 0:00:00
175.4 MiB [] 0% 0.0s175.4 MiB [] 0% 179.9s175.4 MiB [] 0% 636.7s175.4 MiB [] 0% 525.2s175.4 MiB [] 0% 730.0s175.4 MiB [] 0% 641.9s175.4 MiB [] 0% 579.0s175.4 MiB [] 0% 531.8s175.4 MiB [] 0% 493.8s175.4 MiB [] 0% 508.3s175.4 MiB [] 0% 453.4s175.4 MiB [] 0% 434.9s175.4 MiB [] 0% 417.4s175.4 MiB [] 0% 402.3s175.4 MiB [] 0% 371.6s175.4 MiB [] 0% 361.3s175.4 MiB [] 0% 343.9s175.4 MiB [] 0% 326.6s175.4 MiB [] 0% 307.4s175.4 MiB [] 0% 290.4s175.4 MiB [] 0% 276.3s175.4 MiB [] 0% 268.0s175.4 MiB [] 0% 256.8s175.4 MiB [] 0% 247.3s175.4 MiB [] 0% 232.9s175.4 MiB [] 0% 215.7s175.4 MiB [] 0% 200.8s175.4 MiB [] 0% 188.2s175.4 MiB [] 0% 177.1s175.4 MiB [] 0% 171.5s175.4 MiB [] 0% 163.2s175.4 MiB [] 0% 158.3s175.4 MiB [] 0% 147.0s175.4 MiB [] 0% 140.3s175.4 MiB [] 0% 131.9s175.4 MiB [] 0% 124.6s175.4 MiB [] 0% 117.1s175.4 MiB [] 0% 109.9s175.4 MiB [] 0% 104.8s175.4 MiB [] 1% 99.3s175.4 MiB [] 1% 95.0s175.4 MiB [] 1% 89.8s17